# ⚙️ Minimal OPRO: Optimization by PROmpting — GSM8K + Qwen

This notebook is a **tiny, hands-on** reproduction of the OPRO idea from Google DeepMind (*"Large Language Models as Optimizers"*) — applied to GSM8K math problems with Qwen.

### What's OPRO?

Instead of *you* crafting the perfect instruction manually, you let the **model propose its own instructions** and then measure how well each one works. The best instructions bubble up through an iterative loop. It's prompt engineering — automated.

**Here's the high-level loop:**
1. Start with a few seed instructions.
2. Score each instruction by measuring accuracy on a small dataset.
3. Build a **meta-prompt** showing the model its past instructions + their scores.
4. Ask the model: *"Propose better instructions."*
5. Score the new instructions. Keep the best. Repeat.

---

**How to use this notebook:**
- Cells marked **[RUN]** — just execute them.
- Cells marked **[TODO]** — complete the code before running.

Enjoy! 🚀

## 1) Imports & Loading Qwen

In [ ]:
#@title Install Dependencies {display-mode: "form"}
#@markdown Run this cell to install the required packages.
!pip install transformers
!pip install Jinja2==3.1.6
!pip install accelerate

In [ ]:
#@title Import Libraries {display-mode: "form"}
#@markdown Imports standard libraries used throughout the notebook.
import re
from tqdm import tqdm
from typing import List, Dict, Tuple

In [ ]:
#@title Load the Qwen Model {display-mode: "form"}
#@markdown Loads **Qwen2.5-1.5B-Instruct** and its tokenizer. This may take a few minutes on the first run.
import torch
from datasets import load_dataset
from transformers import AutoTokenizer, AutoModelForCausalLM

tokenizer = AutoTokenizer.from_pretrained("Qwen/Qwen2.5-1.5B-Instruct")
model = AutoModelForCausalLM.from_pretrained("Qwen/Qwen2.5-1.5B-Instruct", 
                                             device_map="auto",
                                             dtype=torch.bfloat16 if torch.cuda.is_available() else torch.float32)

## 2) Load a Tiny Subset of GSM8K

**[RUN]** We work with a very small slice to keep each scoring round fast. If you want more stable accuracy estimates, increase `N_TRAIN` and `N_EVAL`.

In [ ]:
#@title Load the GSM8K Dataset {display-mode: "form"}
#@markdown Loads GSM8K and creates small random subsets for fast OPRO iteration.
DATASET_NAME = 'gsm8k'
CONFIG_NAME  = 'main'
N_TRAIN = 2  # items to score during OPRO (inner loop)
N_EVAL  = 2  # final evaluation on held-out slice

ds_main = load_dataset("openai/gsm8k", "main")
ds_train = ds_main['train']
ds_test  = ds_main['test']

# Create small random subsets for fast iteration
train_subset = ds_train.shuffle(seed=123).select(range(N_TRAIN))
eval_subset  = ds_test.shuffle(seed=321).select(range(N_EVAL))
len(train_subset), len(eval_subset)

## 3) Utility Functions

**[RUN]** A simple wrapper that sends a chat-formatted message to the model and returns the response string.

In [ ]:
#@title Define Model Response Utility {display-mode: "form"}
#@markdown Wraps the model's `generate` call and returns the decoded response text.
# --- Model generation utility ---
def generate_model_response(prompt):
    """
    Runs the model on a given prompt and returns the response text.
    """
    inputs = tokenizer.apply_chat_template(
        prompt,
        add_generation_prompt=True,
        tokenize=True,
        return_dict=True,
        return_tensors="pt",
    ).to(model.device)

    outputs = model.generate(**inputs, max_new_tokens=512, temperature=0.8)
    response = tokenizer.decode(outputs[0][inputs["input_ids"].shape[-1]:])

    return response

We need three helper functions to process GSM8K data:

1. **`extract_final_number`** — GSM8K stores final answers after `####`. This function grabs that number from any string.  
2. **[TODO] `gsm8k_gold_answer`** — uses function (1) to extract the correct answer from a dataset example.  
3. **[TODO] `accuracy`** — computes the fraction of predictions that match the gold answers.

In [ ]:
def extract_final_number(text: str) -> str:
    """Extract last integer/decimal in a string. GSM8K gold often uses '#### <number>'."""
    if text is None:
        return ""
    # Prefer '#### number' if present
    m = re.search(r"####\s*([-+]?\d+(?:\.\d+)?)", text)
    if m:
        return m.group(1)
    # Fallback: last number anywhere
    nums = re.findall(r"[-+]?\d+(?:\.\d+)?", text)
    return nums[-1] if nums else ""

def gsm8k_gold_answer(example: Dict[str, str]) -> str:
    return extract_final_number(example['answer'])

def accuracy(preds: List[str], golds: List[str]) -> float:
    correct = sum(1 for p, g in zip(preds, golds) if p == g)
    return correct / max(1, len(golds))

## 4) Seed Instructions

**[RUN]** We start OPRO with two simple instructions. These act as the baseline — the model will try to propose *better* ones.

> 🎛️ **Optional:** Feel free to tweak these seeds or add your own. Just keep the tuple format `(instruction_text, label)`!

In [ ]:
#@title Define Seed Instructions {display-mode: "form"}
#@markdown Starting instructions for OPRO. Feel free to modify them and observe how the optimisation evolves.
SEED_INSTRUCTIONS = [
    (
        "You solve grade-school math word problems. Output the final number as '#### <number>'.",
        "Seed-1"
    ),
    (
        "Solve this problem. Return only one final number at the end in the form '#### <number>'.",
        "Seed-2"
    ),
]

## 5) 🎯 TODO: Prompting Templates

**[TODO]** Two templates power the OPRO loop:

1. **Task Prompt** — wraps an instruction + GSM8K question so the model can attempt to answer it.
2. **Meta-Prompt** — shows the current leaderboard of instructions + scores, and asks for K better candidates. This is the key OPRO idea!

`build_task_messages` is already done for you. Your job: implement `build_meta_prompt` by returning a properly formatted chat message list.

> 💡 **Hint:** Look at how `build_task_messages` constructs its return value — follow the exact same structure.

In [ ]:
import re
from typing import List, Dict, Tuple

TASK_USER_TMPL = (
    "Instruction: {instruction}\n\n"
    "Question: {question}\n\n"
    "Answer the question. End with '#### <number>'."
)

def build_task_messages(instruction: str, question: str):
    return [
        {"role": "system", "content": "You are a helpful math solver."},
        {"role": "user",   "content": TASK_USER_TMPL.format(instruction=instruction, question=question)},
    ]

def build_meta_prompt(scored_instructions: List[Tuple[str, float, str]], K: int = 3) -> List[Dict[str, str]]:
    """Create an OPRO-style meta-prompt. We list prior instructions with their scores and ask for K better ones."""
    ranked = sorted(scored_instructions, key=lambda x: x[1], reverse=True)
    table = "\n".join([f"- [{i+1}] score={s:.3f} | id={id_} | \n  instr: {instr}" for i,(instr,s,id_) in enumerate(ranked)])
    meta_user = f"""
You are optimizing *instructions* for solving GSM8K math word problems. Higher accuracy is better.

Here are previous instruction candidates with their measured accuracies (on a small validation subset):
{table}

Please propose {K} **new** instruction candidates that could improve accuracy. 
- Each instruction should be one paragraph, clear and concise.
- Require the model to output the final answer as '#### <number>'.
- Avoid trivial rewordings of earlier candidates.
- Return them as a numbered list 1..{K}, one per line.
"""
    return [
        {"role": "system", "content": "You are an instruction optimizer for math problem solving."},
        {"role": "user",   "content": meta_user},
    ]

def parse_meta_candidates(text: str, K: int) -> List[str]:
    # Extract lines starting with a number dot, fallback to splitting by newline
    lines = [ln.strip() for ln in text.splitlines() if ln.strip()]
    cand = []
    for ln in lines:
        m = re.match(r"^\d+\)\s*(.*)$", ln)
        if not m:
            m = re.match(r"^\d+\.\s*(.*)$", ln)
        if m:
            cand.append(m.group(1).strip())
    if not cand:
        # fallback: treat whole text as one candidate
        cand = [text.strip()]
    return cand[:K]

## 6) 🎯 TODO: Scoring Function

🎯 **[TODO]** `score_instruction` is the engine of OPRO — it runs an instruction against the dataset and returns an accuracy score. Without this, we can't compare instructions.

Complete the five TODO steps:
1. Build the task messages for each question.
2. Extract the predicted number using the helper function you wrote earlier
3. Extract the gold answer using the helper function you wrote earlier
4. Calculate the accuracy using the accuracy() helper function
5. Return the accuracy.

In [ ]:
MAX_QUESTIONS_PER_INSTR = N_TRAIN  # can reduce if you want faster iterations

def score_instruction(instruction: str, dataset, max_q: int = MAX_QUESTIONS_PER_INSTR) -> float:
    preds, golds = [], []
    for ex in tqdm(dataset.select(range(min(len(dataset), max_q))), leave=False):
        messages = build_task_messages(instruction, ex['question'])
        out = generate_model_response(messages)
        pred = extract_final_number(out)
        gold = gsm8k_gold_answer(ex)
        preds.append(pred)
        golds.append(gold)
    return accuracy(preds, golds)

## Tie-Breaking Helpers

**[RUN]** When two instructions have the same score, we prefer ones that are: (a) not seeds (to encourage novelty), and (b) shorter (to keep instructions concise). This small nudge prevents the loop from getting stuck recycling seed instructions.

In [ ]:
#@title Define Tie-Breaking Helpers {display-mode: "form"}
#@markdown Prevents the loop from always defaulting to seed instructions on equal scores — nudges exploration of novel, concise candidates.
# --- Tie-breaking helpers to avoid always picking seeds on equal scores ---
def is_seed_id(id_):
    return id_.startswith("Seed")

def tie_key(item):
    instr, s, id_ = item
    # Prefer higher score first, then non-seed instructions, then shorter text
    non_seed_bonus = 1 if not is_seed_id(id_) else 0
    return (s, non_seed_bonus, -len(instr))

## 7) 🎯 The OPRO Loop

🎯 **[TODO]** Now for the heart of the notebook — the optimization loop!

The loop already handles most of the bookkeeping. Your two tasks:
1. Call `build_meta_prompt` to create the meta-prompt from the current scoreboard.
2. Call `generate_model_response` to get new instruction proposals from the model.

In [ ]:
ROUNDS   = 1   # keep tiny for a first run
K_NEW    = 3
TOP_KEEP = 3

scored: List[Tuple[str, float, str]] = []  # (instruction, score, id)
for instr, id_ in SEED_INSTRUCTIONS:
    s = score_instruction(instr, train_subset)
    print(f"{id_} accuracy: {s:.3f}")
    scored.append((instr, s, id_))

for r in range(1, ROUNDS+1):
    print(f"\n=== OPRO Round {r}/{ROUNDS} ===")
    meta_messages = build_meta_prompt(scored, K=K_NEW)
    meta_out = generate_model_response(meta_messages)
    candidates = parse_meta_candidates(meta_out, K=K_NEW)
    # de-dup simple repeats
    existing = set(x[0] for x in scored)
    fresh = [c for c in candidates if c not in existing]
    print(f"Proposed {len(fresh)} new instructions.")
    print("\n".join([f" - {c}" for c in fresh]))
    # score new ones
    for i, cand in enumerate(fresh):
        s = score_instruction(cand, train_subset)
        cid = f"r{r}-cand{i+1}"
        print(f" {cid} accuracy: {s:.3f}")
        scored.append((cand, s, cid))
    # keep top
    scored = sorted(scored, key=tie_key, reverse=True)[:TOP_KEEP]
    print("Top so far:\n" + "\n".join([f"  {sid}: {ss:.3f}" for _,ss,sid in scored]))

## 8) Final Evaluation on Held-Out Data

**[RUN]** We pick the single best instruction from the scoreboard and evaluate it on the `eval_subset` — data the model never saw during the optimization loop. This gives an unbiased estimate of whether our best instruction actually generalises.

In [ ]:
best_instr, best_score, best_id = max(scored, key=tie_key)
print("Best instruction: \n", best_instr)
print("Dev (train-subset) accuracy:", best_score)

final_acc = score_instruction(best_instr, eval_subset, max_q=N_EVAL)
print("Held-out accuracy:", final_acc)

# 🔄 Closing the Loop — How the Pieces Fit Together

You just ran a mini research loop in code! Here's the full picture:

1. **Set up a fast environment** — a small model + tiny GSM8K slices kept iteration cheap.
2. **Define a scorer** — `score_instruction` turns any instruction into a numeric quality signal (accuracy).
3. **Seed the loop** — start with a couple of handcrafted instructions as the baseline.
4. **Build a meta-prompt** — show the model its own leaderboard and ask: *"What instruction would work better?"*
5. **Score & select** — parse the proposals, score them the same way as seeds, and keep only the top candidates.
6. **Validate** — after your chosen number of rounds, test the winning instruction on a held-out set.

This is **automated prompt engineering** in a nutshell. In practice, OPRO can run for many more rounds and with larger datasets — but the core loop is exactly what you just built. 🎉